# 02b. Preprocesamiento (version amplia)

Variante menos estricta de `02_preprocessing.ipynb`. Mantiene muchos mas datos:

- Lee **todas** las reviews (sin cap de 2M)
- **Sin** restriccion a top-5 ciudades
- `min_reviews_per_business` 20 -> 10
- Mantiene foto>=5 y 5-core de usuario para no romper la comparabilidad multimodal

Escribe a `data/processed/` igual que el notebook original (sobrescribe). Para volver al dataset estricto, recorre `02_preprocessing.ipynb`.

In [1]:
import sys
sys.path.append('..')

import os
import pandas as pd
from src.data_loader import build_dataset

PROCESSED_DIR = '../data/processed'

## 1. Cargar dataset filtrado (umbrales relajados)

- Minimo 10 reviews por restaurante
- Minimo 5 fotos por restaurante
- `max_review_rows=None` -> lee las 6.99M reviews completas

In [2]:
restaurants, reviews, photos = build_dataset(
    min_reviews_per_business=10,
    min_photos_per_business=5,
    max_review_rows=None,  # leer todo el dataset
)
print(f'Restaurantes: {len(restaurants)} | Reviews: {len(reviews)} | Fotos: {len(photos)}')

Loading businesses: 150346it [00:01, 127340.74it/s]


Restaurants loaded: 52286


Loading reviews: 6990280it [00:52, 133854.53it/s]


Reviews loaded: 4665340
Photos metadata loaded: 164238

Final dataset: 10230 restaurants | 2578367 reviews | 129106 photos
Restaurantes: 10230 | Reviews: 2578367 | Fotos: 129106


## 2. Limpiar texto de reviews

In [3]:
import re

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)  # urls
    text = re.sub(r'[^a-z0-9\s.,!?\'\-]', ' ', text)  # caracteres raros
    text = re.sub(r'\s+', ' ', text).strip()
    return text

reviews['text_clean'] = reviews['text'].apply(clean_text)
reviews['text_len'] = reviews['text_clean'].str.split().str.len()
# Filtrar reviews demasiado cortas (< 10 palabras)
reviews = reviews[reviews['text_len'] >= 10]
print(f'Reviews after text filter: {len(reviews)}')

Reviews after text filter: 2573127


## 3. Filtrar usuarios con poca actividad (5-core)

In [4]:
MIN_USER_REVIEWS = 5
user_counts = reviews['user_id'].value_counts()
active_users = user_counts[user_counts >= MIN_USER_REVIEWS].index
reviews = reviews[reviews['user_id'].isin(active_users)]
print(f'Reviews after user filter: {len(reviews)} | Usuarios: {reviews["user_id"].nunique()}')

Reviews after user filter: 1254538 | Usuarios: 101557


## 4. Recortar restaurantes y fotos a los que sobreviven

Tras el 5-core algunos restaurantes pueden quedar sin reviews. Los sincronizamos.

In [5]:
surviving_ids = set(reviews['business_id'])
restaurants = restaurants[restaurants['business_id'].isin(surviving_ids)]
photos = photos[photos['business_id'].isin(surviving_ids)]
print(f'Final: {len(restaurants)} restaurantes | {len(reviews)} reviews | {len(photos)} fotos')

Final: 10227 restaurantes | 1254538 reviews | 129087 fotos


## 5. Guardar datos procesados

In [6]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
restaurants.to_csv(f'{PROCESSED_DIR}/restaurants.csv', index=False)
reviews.to_csv(f'{PROCESSED_DIR}/reviews.csv', index=False)
photos.to_csv(f'{PROCESSED_DIR}/photos.csv', index=False)
print('Saved to data/processed/')

Saved to data/processed/
